# Explicamelo como si tuviera 5 años


![Architecture](images/architecture.png)

En este cuaderno vamos a recorrer el proceso de configurar un chatbot simple en LangChain.

A lo largo de este proceso, vamos a mostrar cómo LangSmith puede usarse para mejorar la experiencia del desarrollador en aplicaciones de IA.

Empecemos cargando nuestras variables de entorno desde el archivo .env.

In [1]:
from dotenv import load_dotenv
load_dotenv(dotenv_path=".env", override=True)
# Cargar las siguientes variables de entorno:
# LANGSMITH_TRACING=true
# LANGSMITH_ENDPOINT="https://api.smith.langchain.com"
# LANGSMITH_PROJECT="eli5-bot"
# LANGSMITH_API_KEY="<redacted>"

# OPENAI_API_KEY="<redacted>"
# TAVILY_API_KEY="<redacted>"

True

## Setup

Configuremos una herramienta llamada Tavily para permitir que nuestro asistente busque en la web al responder.

In [2]:
from langchain_community.tools.tavily_search import TavilySearchResults

web_search_tool = TavilySearchResults(max_results=1)

C:\Users\sergi\AppData\Local\Temp\ipykernel_21024\632376592.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults
C:\Users\sergi\AppData\Local\Temp\ipykernel_21024\632376592.py:3: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  web_search_tool = TavilySearchResults(max_results=1)


Diseñemos un prompt para RAG que usaremos a lo largo de todo el cuaderno.

In [3]:
prompt = """Sos un profesor y un experto en explicar temas complejos de una manera fácil de entender.
Tu trabajo es responder la pregunta dada de forma tal que incluso un niño de 5 años pueda comprenderla.
Se te ha proporcionado el contexto necesario para responder la pregunta.

Pregunta: {question} 

Contexto: {context}

Respuesta:"""
print("Prompt Template: ", prompt)

Prompt Template:  Sos un profesor y un experto en explicar temas complejos de una manera fácil de entender.
Tu trabajo es responder la pregunta dada de forma tal que incluso un niño de 5 años pueda comprenderla.
Se te ha proporcionado el contexto necesario para responder la pregunta.

Pregunta: {question} 

Contexto: {context}

Respuesta:


## Creando nuestra aplicación

In [4]:
from openai import OpenAI
from langsmith import traceable
from langsmith.wrappers import wrap_openai

openai_client = wrap_openai(OpenAI())

@traceable
def search(question):
    web_docs = web_search_tool.invoke({"query": question})
    web_results = "\n".join([d["content"] for d in web_docs])
    return web_results
    
@traceable
def explain(question, context):
    formatted = prompt.format(question=question, context=context)
    
    completion = openai_client.chat.completions.create(
        messages=[
            {"role": "system", "content": formatted},
            {"role": "user", "content": question},
        ],
        model="gpt-4o-mini",
    )
    return completion.choices[0].message.content

@traceable
def eli5(question):
    context = search(question)
    answer = explain(question, context)
    return answer

## Testeando nuestra application

In [5]:
question = "Qué es la biotecnología?"
print(eli5(question))

¡Claro! Imagina que la biotecnología es como un superpoder que tienen los científicos para ayudar a la naturaleza a hacer cosas increíbles. 

Cuando hablamos de biotecnología, nos referimos a usar cosas de los seres vivos, como plantas, animales y microbios (que son criaturas muy, muy pequeñas) para crear productos que nos ayudan en nuestra vida diaria. 

Por ejemplo, piensa en el pan que comes. Para hacerlo, a veces se usan unos pequeños seres llamados levaduras, que ayudan a que la masa suba y se vuelva esponjosa. Eso es un pequeño ejemplo de biotecnología. 

También hay biotecnología en la medicina, donde los científicos utilizan células y microorganismos para crear medicinas que nos ayudan a sentirnos mejor. 

Así que, en resumen, ¡la biotecnología es usar la vida para hacer cosas que nos ayudan a vivir mejor!
